# Prueba base · Llama en inglés sobre el piloto

Corre los 80 prompts del piloto **en inglés original** sobre Llama 3.1 8B Instruct y mide qué porcentaje de ataques funciona (ASR).
Sirve para saber, antes de traducir todo, si Llama cae con suficientes ataques como para comparar inglés contra español.

**Antes de ejecutar:**
1. **GPU:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*.
2. **Acceso a Llama:** en https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct, con la misma cuenta del token, completa el formulario y espera la aprobación (llega por correo).
3. **Secreto `HF_TOKEN`** en la llave de la barra izquierda, con *Acceso al notebook* activado.
4. Haber corrido la etapa 1 (`pilot.parquet` en Drive).

In [ ]:
# Solo en Colab: montar Drive, clonar el repo e instalar
import sys

REPO = "https://github.com/santiaagoe/hackaprompt-cl.git"
BRANCH = "main"  # para probar una rama sin fusionar, poner aquí su nombre
DRIVE = "/content/drive/MyDrive/hackaprompt-cl"

if "google.colab" in sys.modules:
    import os

    from google.colab import drive, userdata

    drive.mount("/content/drive")
    %cd /content
    if not os.path.exists("hackaprompt-cl"):
        !git clone -q -b {BRANCH} {REPO}
    %cd hackaprompt-cl
    !git pull -q
    !pip install -q -e ".[evaluate]"
    # pip install -e no es visible hasta reiniciar el entorno; se agrega la ruta a mano
    if "/content/hackaprompt-cl/src" not in sys.path:
        sys.path.insert(0, "/content/hackaprompt-cl/src")
    os.environ["HF_TOKEN"] = (userdata.get("HF_TOKEN") or "").strip()
    os.environ["HACKAPROMPT_DATA_DIR"] = f"{DRIVE}/data"
    os.environ["HACKAPROMPT_RESULTS_DIR"] = f"{DRIVE}/results"
    os.makedirs(f"{DRIVE}/results", exist_ok=True)

In [ ]:
# Debe mostrar una GPU (Tesla T4 o similar)
!nvidia-smi -L

## Correr los ataques

La primera vez descarga Llama (unos 16 GB, varios minutos). Si Colab se corta, vuelve a ejecutar desde arriba: retoma donde quedó.

In [ ]:
!python -m hackaprompt_cl.evaluate.run --split pilot --condition en

## Resultados

In [ ]:
from pathlib import Path

import pandas as pd

from hackaprompt_cl.config import RESULTS_DIR
from hackaprompt_cl.evaluate.run import asr_table

pd.set_option("display.max_colwidth", 150)
path = sorted(Path(RESULTS_DIR).glob("pilot_en_*.parquet"))[-1]
results = pd.read_parquet(path)
print(path.name)
asr_table(results)

In [ ]:
# Ataques que funcionaron (criterio estricto)
results[results["strict"]][["id", "level", "completion"]].head(10)

In [ ]:
# Ataques que fallaron: ¿Llama se negó, respondió otra cosa o casi lo logró?
results[~results["strict"]][["id", "level", "relaxed", "completion"]].head(15)

## Cómo leer el resultado

Miren la fila **total**, columna `strict` (ASR estricto, comparable con el paper):

| ASR estricto en inglés | Qué significa | Qué hacer |
| --- | --- | --- |
| 30% o más | Hay suficientes ataques exitosos para comparar | Seguir con el plan |
| 15% a 30% | Comparación posible pero con menos poder estadístico | Considerar subir la muestra o usar también el criterio relajado |
| Menos de 15% | Llama resiste casi todos los ataques en inglés | Decidir en grupo antes de traducir (ver issue) |

Pegar la tabla de `asr_table` en el issue de la prueba base.